# 07 · Função de distribuição radial (RDF) e validação do modelo de água

| | |
|---|---|
| **Autor** | Patrick Allan dos Santos Faustino |
| **Projeto** | Revisão tutorial de dinâmica molecular — xilanase GH11 (PDB 1XNB) |
| **Versão** | v2026.11 — notebook autocontido |

**Pergunta.** A água da simulação tem a estrutura local esperada — posição e altura do 1º pico de g$_{OO}$(r) e número de coordenação — comparada com a difração de raios X?

$g(r)$ entre dois grupos, com o número de coordenação $n(r)$ por **contagem direta de pares**.
Não é série temporal: é um perfil em $r$, e a incerteza de cada bin vem da dispersão entre réplicas
(ou entre blocos, com uma réplica só).

| Configuração | Grupo 1 | Grupo 2 | Para quê |
|---|---|---|---|
| **(a) validação da água** (ativa) | `water and name O` | `water and name O` | comparar com difração (Skinner *et al.*, *J. Chem. Phys.* **2013**, 138, 074506) |
| (b) hidratação da proteína | `protein and name O` | `water and name O` | didática; sem contrapartida experimental direta |

> O MDTraj renomeia a água do GROMACS ao ler o `.gro` (SOL → HOH, OW → O, HW1 → H1); o sítio
> virtual dos modelos de 4 pontos (MW) fica sem elemento. `water and name O` funciona nas duas
> convenções.
>
> **Só águas do bulk como referência.** Numa caixa com proteína, a esfera de raio $r$ em volta de
> uma água próxima da proteína inclui volume ocupado por ela, e a cauda de g(r) fica abaixo de 1
> (volume excluído). Por isso, a cada frame, as águas do grupo 1 a menos de
> `DISTANCIA_EXCLUSAO_NM` (1,5 nm) de qualquer átomo pesado da proteína ficam de fora; o grupo 2
> continua com todas as águas. Com 1,5 nm de distância e raio máximo de 1,0 nm, nenhuma esfera
> alcança a proteína.
>
> **Água pura do mesmo modelo (notebook 14).** A curva da caixa de OPC pura, nas mesmas condições,
> separa o que é **do modelo** (a altura do pico, o número de coordenação) do que é efeito da
> proteína ou da análise. A água do bulk do sistema deve reproduzir a da caixa pura; o modelo é que
> se compara com a difração.
>
> **Modelos rígidos não polarizáveis superestruturam a água.** O desacordo na altura do 1º pico é
> conhecido e não é erro seu; posição e cauda vêm antes da altura, e o número de coordenação
> (4,3–4,7 por difração) é o teste mais discriminante.

> **Pré-processamento** (a PBC é tratada no GROMACS, não em Python):
> ```bash
> gmx trjconv -s md_300ns.tpr -f md_300ns.xtc -o md_whole.xtc -pbc whole
> gmx trjconv -s md_300ns.tpr -f md_whole.xtc -o md_300ns_mol.xtc -pbc mol -center -ur compact
> ```

**Blocos:** 1 bibliotecas · 2 configuração e dados · 3 cálculos · 4 estatística ·
5 gráfico (matplotlib) · 6 gráfico interativo (Plotly) · 7 exportação.

## 1 · Bibliotecas

In [ ]:
import sys
import platform
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import mdtraj as md
import MDAnalysis as mda
from scipy.signal import find_peaks
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

INICIO = datetime.now()
VERSOES = {
    "Python": platform.python_version(),
    "NumPy": np.__version__,
    "pandas": pd.__version__,
    "SciPy": scipy.__version__,
    "Matplotlib": matplotlib.__version__,
    "Plotly": plotly.__version__,
    "MDTraj": md.__version__,
    "MDAnalysis": mda.__version__,
}
print(f"Data e hora da execução: {INICIO:%d/%m/%Y %H:%M:%S}")
print(f"Sistema operacional: {platform.platform()}")
df_versoes = pd.DataFrame(list(VERSOES.items()), columns=["biblioteca", "versao"])
df_versoes

## 2 · Configuração da análise e carregamento dos dados

Aqui só se leem a topologia e alguns frames (caixa); a trajetória é percorrida em lotes no
bloco 3.

In [ ]:
# ── Onde estão as simulações ────────────────────────────────────────────────
PASTA_BASE = Path("..")             # pasta que contém rep1/, rep2/, ...   <- AJUSTE
SISTEMA = "xilanase_1XNB"           # nome da pasta de resultados
REPLICAS = [1, 2, 3, 4, 5]          # uma réplica só também funciona: [1]
PREFIXO_MD = "md_300ns"             # -deffnm da produção


def pasta_replica(n):
    """Pasta da réplica n."""
    return PASTA_BASE / f"rep{n}"

ARQUIVO_TOP = f"{PREFIXO_MD}.gro"            # sistema inteiro
ARQUIVO_TRAJ = f"{PREFIXO_MD}_mol.xtc"        # sistema inteiro, -pbc mol -center
ARQUIVO_TPR = f"{PREFIXO_MD}.tpr"            # elementos pelo nº atômico (MDAnalysis >= 2.11); None = pelo nome
ARQUIVOS = {"topologia": ARQUIVO_TOP, "trajetória": ARQUIVO_TRAJ, **({"tpr": ARQUIVO_TPR} if ARQUIVO_TPR else {})}

# ── Amostragem ──────────────────────────────────────────────────────────────
DT_ANALISE_PS = 500.0           # g(r) de frames próximos é quase igual: 2 000–5 000 frames bastam
DESCARTE_NS = 0.0
NIVEL_CONFIANCA = 0.95
N_BLOCOS = 5                    # barra de erro por blocos quando há UMA réplica

# ── Grupos e binagem ────────────────────────────────────────────────────────
SELECAO_GRUPO_1 = "water and name O"     # referência
SELECAO_GRUPO_2 = "water and name O"     # espécie distribuída
MAX_ATOMOS_GRUPO_1 = 500        # subamostra da referência (não enviesa g(r), só reduz a precisão); None = todos
SEMENTE = 20260905
LARGURA_BIN_NM = 0.002
RAIO_MAX_NM = 1.0               # limitado a metade da menor ALTURA da caixa
MEMORIA_LOTE_MB = 400           # teto do bloco de distâncias em RAM

# ── Águas de referência só no bulk (por frame) ──────────────────────────────
SELECAO_EXCLUSAO = "protein and not element H"   # None = sem filtro (use None na configuração b)
DISTANCIA_EXCLUSAO_NM = 1.5     # água do grupo 1 mais perto que isso da proteína fica de fora no frame
                                # (≥ RAIO_MAX_NM: a esfera de cada referência não alcança a proteína)

# ── Normalização ────────────────────────────────────────────────────────────
# "caixa"   -> ρ = N2_ef/<V> (ou DENSIDADE_REFERENCIA_NM3): a definição de g(r), a do `gmx rdf`
# "ganguly" -> caixa + correção de Ganguly & van der Vegt (JCTC 2013, 9, 1347) do reservatório finito
# "casca"   -> força a casca externa a 1 (só para figura; inválido para integrais de KB)
NORMALIZACAO = "caixa"
FRACAO_CASCA = 0.25
# Caixa com soluto: N2/V_caixa subestima a densidade do solvente; imponha a do bulk do SEU modelo
# de água (caixa pura, mesmas T e p). 33,3 nm⁻³ só vale se o modelo reproduz a experimental.
DENSIDADE_REFERENCIA_NM3 = 33.3311          # None = N2/<V_caixa>

# ── Referência experimental ─────────────────────────────────────────────────
REFERENCIA_EXPERIMENTAL = {"rotulo": "Skinner et al. (2013)", "arquivo": None,    # CSV opcional r_nm, g
                           "r_pico_nm": 0.280, "u_r_pico_nm": 0.001, "g_pico": 2.57, "u_g_pico": 0.05}
COORDENACAO_EXP = (4.3, 4.7)             # faixa típica por difração
COMPARAR_EXPERIMENTO = True              # só faz sentido na configuração (a), água–água
# Água pura do MESMO modelo, mesmas condições (saída do notebook 14); None = sem essa referência
REFERENCIA_MODELO = {"rotulo": "OPC pura (notebook 14)",
                     "arquivo": Path("resultados") / "agua_OPC" / "14_caixa_agua" / "caixa_agua_dados.csv",
                     "r_pico_nm": 0.281, "g_pico": 3.2208, "u_g_pico": 0.0063, "numero_coordenacao": 4.688}
RAIO_FIGURA_NM = None                    # None = 4 × posição do 1º mínimo

# ── Saída ───────────────────────────────────────────────────────────────────
DIR_SAIDA = Path("resultados") / SISTEMA / "07_rdf"
PREFIXO = "rdf"
TITULO = "07 · RDF e validação do modelo de água"

In [ ]:
linhas = []
for n in REPLICAS:
    for papel, nome in ARQUIVOS.items():
        caminho = pasta_replica(n) / nome
        linhas.append({"replica": f"rep{n}", "arquivo": papel, "caminho": str(caminho),
                       "existe": caminho.exists(),
                       "tamanho_MB": round(caminho.stat().st_size / 1024 ** 2, 1) if caminho.exists() else np.nan})
df_arquivos = pd.DataFrame(linhas)
display(df_arquivos)
if not df_arquivos["existe"].all():
    raise FileNotFoundError("Arquivos ausentes (veja a tabela acima). Ajuste RAIZ/SISTEMA/REPLICAS.")

In [ ]:
MAPA_RESNAMES = {"HIE": "HIS", "HID": "HIS", "HIP": "HIS", "HSD": "HIS", "HSE": "HIS",
                 "HSP": "HIS", "CYX": "CYS", "CYM": "CYS", "LYN": "LYS", "ASH": "ASP",
                 "GLH": "GLU"}


def tpr_de(n):
    """Caminho do .tpr da réplica n (None se ARQUIVO_TPR = None)."""
    return pasta_replica(n) / ARQUIVO_TPR if ARQUIVO_TPR else None


def elementos_do_tpr(tpr, top, arquivo):
    """Elemento de cada átomo da topologia, pelo número atômico gravado no .tpr.

    O .gro pode ser o sistema inteiro ou um subconjunto na MESMA ordem (o _prot.gro).
    O casamento é resíduo a resíduo, pela sequência de nomes de átomos: o MDAnalysis
    renumera os resíduos do .tpr a partir de 1, então o nº do resíduo (Leu4...) não serve.
    """
    u = mda.Universe(str(tpr))
    if not hasattr(u.atoms, "elements"):
        raise ValueError("o .tpr não trouxe números atômicos")
    if str(arquivo).endswith(".gro"):            # nomes crus (o MDTraj renomeia OW -> O etc.)
        linhas = Path(arquivo).read_text().splitlines()[2:2 + top.n_atoms]
        nomes = [l[10:15].strip() for l in linhas]
    else:
        nomes = [a.name for a in top.atoms]
    cortes = np.flatnonzero(np.diff(u.atoms.resindices)) + 1
    residuos_tpr = list(zip(np.split(u.atoms.names, cortes), np.split(u.atoms.elements, cortes)))
    elementos, j = [], 0
    for residuo in top.residues:
        alvo = [nomes[a.index] for a in residuo.atoms]
        while j < len(residuos_tpr) and list(residuos_tpr[j][0]) != alvo:
            j += 1
        if j == len(residuos_tpr):
            raise ValueError(f"{residuo} não casa com o .tpr (o .gro não é um subconjunto ordenado dele)")
        elementos += list(residuos_tpr[j][1])
        j += 1
    return elementos


def ler_topologia(arquivo, tpr=None):
    """Topologia MDTraj de um .gro/.pdb, com os elementos certos.

    * Com `tpr` (MDAnalysis >= 2.11 lê o .tpr do GROMACS 2026): o elemento vem do número
      atômico gravado no .tpr — a mesma informação que o GROMACS usa.
    * Sem `tpr` (ou se a leitura falhar): o MDTraj deduz o elemento pelo nome do átomo, e o
      FE do heme viraria flúor (19 u em vez de 55,85 u); ele é corrigido e um aviso é impresso.
    * Resíduos com nome de protonação (HIE, CYM...) recebem o nome padrão, para que
      `protein` e `resname` funcionem.
    """
    top = md.load_topology(str(arquivo))
    for residuo in top.residues:
        residuo.name = MAPA_RESNAMES.get(residuo.name.upper(), residuo.name)
    if tpr is not None and Path(tpr).exists():
        try:
            elementos = elementos_do_tpr(tpr, top, arquivo)
            for atomo, simbolo in zip(top.atoms, elementos):
                atomo.element = md.element.get_by_symbol(simbolo.capitalize()) if simbolo else md.element.virtual
            return top
        except Exception as erro:
            print(f"[aviso] .tpr não usado ({erro}); elementos deduzidos pelo nome do átomo.")
    for atomo in top.atoms:
        if atomo.residue.name == "HEM" and atomo.name.upper() == "FE":
            atomo.element = md.element.iron
    return top


def selecionar(top, consulta, n=None):
    """top.select com erro explícito se vazio (ou se não tiver n átomos)."""
    idx = top.select(consulta)
    if len(idx) == 0:
        raise ValueError(f"Seleção vazia: '{consulta}' (no MDTraj o nº do resíduo é `resSeq`).")
    if n is not None and len(idx) != n:
        raise ValueError(f"'{consulta}' casou com {len(idx)} átomos (esperado {n}).")
    return idx


def intervalo_do_arquivo_ps(xtc, top):
    """Δt entre frames gravados no .xtc (lê só os 2 primeiros frames)."""
    for lote in md.iterload(str(xtc), top=top, chunk=2, atom_indices=[0]):
        return float(np.diff(lote.time)[0])


def estride(xtc, top, dt_alvo_ps):
    """Passo de leitura que leva o Δt do arquivo ao Δt de análise."""
    dt = intervalo_do_arquivo_ps(xtc, top)
    passo = max(1, int(round(dt_alvo_ps / dt)))
    if dt_alvo_ps < dt:
        print(f"[aviso] Δt pedido {dt_alvo_ps:g} ps é menor que o do arquivo ({dt:g} ps).")
    elif abs(passo * dt - dt_alvo_ps) > 1e-6 * dt_alvo_ps:
        print(f"[aviso] Δt {dt_alvo_ps:g} ps não é múltiplo de {dt:g} ps; usando {passo * dt:g} ps.")
    return passo, dt


def producao(t_ns):
    """Máscara dos frames de produção (depois de DESCARTE_NS)."""
    t_ns = np.asarray(t_ns, dtype=float)
    mascara = t_ns >= t_ns[0] + DESCARTE_NS
    if not mascara.any():
        raise ValueError("DESCARTE_NS é maior que a trajetória.")
    return mascara



def alturas_da_caixa(vetores):
    """Distâncias entre faces opostas (nm); em caixa triclínica são menores que os vetores."""
    v = np.asarray(vetores, dtype=float).reshape(-1, 3, 3)
    a, b, c = v[:, 0], v[:, 1], v[:, 2]
    vol = np.abs(np.einsum("ij,ij->i", a, np.cross(b, c)))
    return np.stack([vol / np.linalg.norm(np.cross(b, c), axis=1), vol / np.linalg.norm(np.cross(c, a), axis=1),
                     vol / np.linalg.norm(np.cross(a, b), axis=1)], axis=1)


sistemas, linhas = {}, []
for n in REPLICAS:
    rep, xtc = f"rep{n}", pasta_replica(n) / ARQUIVO_TRAJ
    top = ler_topologia(pasta_replica(n) / ARQUIVO_TOP, tpr_de(n))
    idx1_total, idx2 = selecionar(top, SELECAO_GRUPO_1), selecionar(top, SELECAO_GRUPO_2)
    idx1 = idx1_total
    if MAX_ATOMOS_GRUPO_1 and idx1.size > MAX_ATOMOS_GRUPO_1:
        idx1 = np.sort(np.random.default_rng(SEMENTE).choice(idx1, MAX_ATOMOS_GRUPO_1, replace=False))
    passo, dt = estride(xtc, top, DT_ANALISE_PS)
    with md.open(str(xtc)) as arq:
        n_frames = len(arq)
    alturas = [alturas_da_caixa(md.load_frame(str(xtc), int(i), top=top).unitcell_vectors[0])[0]
               for i in np.unique(np.linspace(0, n_frames - 1, min(20, n_frames)).astype(int))]
    idx_exc = selecionar(top, SELECAO_EXCLUSAO) if SELECAO_EXCLUSAO else np.array([], dtype=int)
    indices = np.union1d(np.union1d(idx1_total, idx2), idx_exc)
    n_ref_max = idx1.size                                         # teto de referências por frame
    sobrepostos = np.intersect1d(idx1_total, idx2).size / idx1_total.size   # 1 se o grupo 1 está no 2
    sistemas[rep] = {"xtc": xtc, "top": top, "indices": indices, "passo": passo,
                     "local1": np.searchsorted(indices, idx1_total), "local2": np.searchsorted(indices, idx2),
                     "local_exc": np.searchsorted(indices, idx_exc),
                     "n2": idx2.size, "n2_efetivo": idx2.size - sobrepostos,
                     "identicos": bool(np.array_equal(idx1_total, idx2)),
                     "lote": max(1, int(MEMORIA_LOTE_MB * 1e6 / (4 * n_ref_max * idx2.size)))}
    linhas.append({"replica": rep, "frames_arquivo": n_frames, "passo": passo, "dt_usado_ps": passo * dt,
                   "grupo1_max_por_frame": n_ref_max, "grupo1_total": idx1_total.size, "grupo2": idx2.size,
                   "exclusao": f"{SELECAO_EXCLUSAO} < {DISTANCIA_EXCLUSAO_NM} nm" if SELECAO_EXCLUSAO else "—",
                   "altura_minima_nm": float(np.min(alturas))})
df_selecao = pd.DataFrame(linhas)

limite = float(df_selecao["altura_minima_nm"].min() / 2)          # imagem mínima, grade única p/ todas
raio_max = min(RAIO_MAX_NM or limite, limite)
if RAIO_MAX_NM and RAIO_MAX_NM > limite:
    print(f"[!] RAIO_MAX_NM reduzido a {limite:.3f} nm (metade da menor altura da caixa).")
n_bins = int(raio_max / LARGURA_BIN_NM)
BORDAS = np.linspace(0.0, n_bins * LARGURA_BIN_NM, n_bins + 1)
CENTROS = 0.5 * (BORDAS[1:] + BORDAS[:-1])
CASCAS = 4.0 / 3.0 * np.pi * (BORDAS[1:] ** 3 - BORDAS[:-1] ** 3)
print(f"Grade: {n_bins} bins de {10 * LARGURA_BIN_NM:.2f} Å até {BORDAS[-1]:.3f} nm | normalização: {NORMALIZACAO}")
df_selecao

## 3 · Cálculos

Uma passada por réplica; cada frame vira uma linha de contagens por bin, o que permite recortar
a equilibração e dividir em blocos sem reler a trajetória. A cada frame, o grupo 1 é refeito: só as
águas a mais de `DISTANCIA_EXCLUSAO_NM` da proteína (`md.compute_neighbors`, com PBC), sorteadas até
`MAX_ATOMOS_GRUPO_1`. Como o número de referências muda de frame para frame, cada histograma é
dividido pelo seu próprio número de referências antes da média. A soma sobre frames é exatamente a
que se obteria com tudo na memória. O $n(r)$ vem da contagem de pares, imune à normalização.

In [ ]:
def g_de_contagem(media_por_referencia, s, volume):
    """Histograma médio POR REFERÊNCIA -> (g, ρ usada)."""
    local = media_por_referencia / CASCAS
    if NORMALIZACAO == "casca":
        rho = float(local[CENTROS > (1 - FRACAO_CASCA) * BORDAS[-1]].mean())
    else:
        rho = DENSIDADE_REFERENCIA_NM3 or s["n2_efetivo"] / volume
    g = local / rho
    if NORMALIZACAO == "ganguly":
        excesso = np.cumsum(4 * np.pi * CENTROS ** 2 * (g - 1) * rho * np.diff(BORDAS))
        disponivel = s["n2"] * (1 - 4 / 3 * np.pi * CENTROS ** 3 / volume)
        denominador = disponivel - excesso - (1.0 if s["identicos"] else 0.0)
        if np.any(denominador <= 0):
            raise ValueError("Ganguly com denominador não positivo: raio grande demais para a caixa.")
        g = g * disponivel / denominador
    return g, rho


sorteio = np.random.default_rng(SEMENTE)
quadros, blocos, volumes, referencias = [], [], {}, {}
for rep, s in sistemas.items():
    contagens, n_refs, tempos, vols, altura_min = [], [], [], [], np.inf
    for lote in md.iterload(str(s["xtc"]), top=s["top"], chunk=s["lote"], stride=s["passo"], atom_indices=s["indices"]):
        if s["local_exc"].size:                                   # águas do grupo 1 perto da proteína, por frame
            perto = md.compute_neighbors(lote, DISTANCIA_EXCLUSAO_NM, s["local_exc"],
                                         haystack_indices=s["local1"], periodic=True)
        for k in range(lote.n_frames):
            refs = np.setdiff1d(s["local1"], perto[k]) if s["local_exc"].size else s["local1"]
            if MAX_ATOMOS_GRUPO_1 and refs.size > MAX_ATOMOS_GRUPO_1:
                refs = np.sort(sorteio.choice(refs, MAX_ATOMOS_GRUPO_1, replace=False))
            histograma = np.zeros(n_bins, dtype=np.float32)
            if refs.size:
                pares = np.column_stack([np.repeat(refs, s["local2"].size), np.tile(s["local2"], refs.size)])
                pares = pares[pares[:, 0] != pares[:, 1]]         # sem o par do átomo consigo
                dk = md.compute_distances(lote[k], pares, periodic=True)[0]
                dk = dk[dk < BORDAS[-1]]
                histograma = np.bincount((dk / LARGURA_BIN_NM).astype(np.int32), minlength=n_bins)[:n_bins].astype(np.float32)
            contagens.append(histograma)
            n_refs.append(refs.size)
        tempos.append(np.asarray(lote.time) / 1000.0)
        vols.append(np.asarray(lote.unitcell_volumes))
        altura_min = min(altura_min, float(alturas_da_caixa(lote.unitcell_vectors).min()))
    contagens, n_refs = np.array(contagens), np.array(n_refs, dtype=float)
    t_ns, vols = np.concatenate(tempos), np.concatenate(vols)
    if altura_min / 2 < BORDAS[-1]:
        print(f"[!] {rep}: a caixa encolheu — limite real {altura_min / 2:.3f} nm < {BORDAS[-1]:.3f} nm. Reduza RAIO_MAX_NM.")
    usar = producao(t_ns) & (n_refs > 0)
    if (producao(t_ns) & (n_refs == 0)).any():
        print(f"[!] {rep}: {int((producao(t_ns) & (n_refs == 0)).sum())} frame(s) sem água no bulk — ignorados.")
    contagens, n_refs, volume = contagens[usar], n_refs[usar], float(vols[usar].mean())
    por_ref = contagens / n_refs[:, None]                         # histograma por referência, frame a frame
    volumes[rep] = volume
    referencias[rep] = {"referencias_por_frame_media": n_refs.mean(), "referencias_por_frame_min": n_refs.min()}
    if SELECAO_EXCLUSAO and n_refs.mean() < 100:
        print(f"[!] {rep}: só {n_refs.mean():.0f} águas de bulk por frame — a caixa tem pouca água longe da proteína. "
              "Reduza DISTANCIA_EXCLUSAO_NM (sem descer de RAIO_MAX_NM) ou aumente DT_ANALISE_PS para mais frames; "
              "a referência limpa do modelo continua sendo a caixa de água pura (notebook 14).")
    g, rho = g_de_contagem(por_ref.mean(axis=0), s, volume)
    quadros.append(pd.DataFrame({"replica": rep, "r_nm": CENTROS, "g": g, "contagem": contagens.sum(axis=0),
                                 "n_acumulado": np.cumsum(por_ref.mean(axis=0))}))
    for b, fatia in enumerate(np.array_split(np.arange(por_ref.shape[0]), N_BLOCOS), start=1):
        blocos.append(pd.DataFrame({"replica": rep, "bloco": b, "r_nm": CENTROS,
                                    "g": g_de_contagem(por_ref[fatia].mean(axis=0), s, volume)[0]}))
    print(f"[{rep}] {int(usar.sum())} frames de produção | {n_refs.mean():.0f} águas de referência por frame "
          f"(mín. {n_refs.min():.0f}) | ρ usada {rho:.3f} nm⁻³ | ρ da caixa {s['n2_efetivo'] / volume:.3f} nm⁻³")
df_selecao = df_selecao.merge(pd.DataFrame(referencias).T.rename_axis("replica").reset_index(), on="replica")
df_rdf = pd.concat(quadros, ignore_index=True)
df_blocos = pd.concat(blocos, ignore_index=True)
df_rdf.groupby("replica", sort=False)["g"].describe()

## 4 · Estatística

Média ± SEM por bin entre réplicas (com 1 réplica, entre blocos) e IC t de Student. Primeiro
pico, primeiro mínimo e número de coordenação, com dois filtros para o pico: **proeminência**
(o ruído cruza g = 1 várias vezes com bins finos) e **significância de Poisson**
($\sigma(g) = g/\sqrt{N}$, com N os pares contados no bin). O mínimo tem limiar próprio, menor.
**A cauda valida tudo:** num líquido homogêneo $g(r) \to 1$; se não vai, a normalização (ou o
raio) está errada e a altura não se compara com a literatura.

In [ ]:
def ineficiencia_estatistica(x, t_minimo=3):
    """g = 1 + 2·τ_int (em frames) — Chodera et al., JCTC 2007, 3, 26.

    ACF não enviesada (FFT), somada até o 1º C(t) <= 0 depois de t_minimo, com o
    peso (1 − t/N) aplicado UMA vez. Igual a
    pymbar.timeseries.statistical_inefficiency(fast=False).
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    if n < 3 or np.allclose(x, x[0]):
        return 1.0
    d = x - x.mean()
    var = float(d @ d) / n
    tamanho = 1 << (2 * n - 1).bit_length()
    f = np.fft.rfft(d, tamanho)
    acf = np.fft.irfft(f * np.conjugate(f), tamanho)[:n] / ((n - np.arange(n)) * var)
    g = 1.0
    for t in range(1, n - 1):
        if acf[t] <= 0.0 and t > t_minimo:
            break
        g += 2.0 * acf[t] * (1.0 - t / n)
    return float(max(g, 1.0))


def t_student(graus_liberdade):
    return stats.t.ppf(0.5 + NIVEL_CONFIANCA / 2, df=max(graus_liberdade, 1))


def resumo_replica(y, dt_ns, observavel, replica, unidade=""):
    """Média, DP, g, N_ef e SEM corrigido de UMA série (uma réplica)."""
    y = np.asarray(y, dtype=float)
    g = ineficiencia_estatistica(y)
    n_ef = max(y.size / g, 2.0)
    dp = float(np.std(y, ddof=1))
    sem = dp / np.sqrt(n_ef)
    media = float(np.mean(y))
    return {"observavel": observavel, "replica": replica, "unidade": unidade,
            "n": y.size, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(n_ef - 1) * sem,
            "ic_superior": media + t_student(n_ef - 1) * sem,
            "minimo": float(y.min()), "maximo": float(y.max()),
            "g": g, "tau_int_ns": (g - 1.0) / 2.0 * dt_ns, "n_efetivo": n_ef}


def entre_replicas(medias, observavel, unidade=""):
    """A réplica é a unidade estatística: média ± SEM das médias das réplicas."""
    medias = np.asarray(medias, dtype=float)
    k = medias.size
    media = float(medias.mean())
    dp = float(np.std(medias, ddof=1)) if k > 1 else np.nan
    sem = dp / np.sqrt(k) if k > 1 else np.nan
    return {"observavel": observavel, "replica": "entre_replicas", "unidade": unidade,
            "n": k, "media": media, "desvio_padrao": dp, "sem": sem,
            "ic_inferior": media - t_student(k - 1) * sem if k > 1 else np.nan,
            "ic_superior": media + t_student(k - 1) * sem if k > 1 else np.nan,
            "minimo": float(medias.min()), "maximo": float(medias.max()),
            "g": np.nan, "tau_int_ns": np.nan, "n_efetivo": float(k)}


def estatisticas_series(df_series, unidades):
    """Por réplica (só produção) + linha 'entre_replicas' quando há 2 ou mais."""
    linhas = []
    prod = df_series[df_series["fase"] == "producao"]
    for obs, bloco in prod.groupby("observavel", sort=False):
        por_rep = [resumo_replica(g["valor"], float(np.median(np.diff(g["tempo_ns"]))),
                                  obs, rep, unidades.get(obs, ""))
                   for rep, g in bloco.groupby("replica", sort=False)]
        linhas += por_rep
        if len(por_rep) > 1:
            linhas.append(entre_replicas([l["media"] for l in por_rep], obs, unidades.get(obs, "")))
    return pd.DataFrame(linhas)


def referencia(df_stats, observavel):
    """Linha 'entre_replicas' do observável (ou a da única réplica)."""
    bloco = df_stats[df_stats["observavel"] == observavel]
    entre = bloco[bloco["replica"] == "entre_replicas"]
    return (entre if len(entre) else bloco).iloc[0]


def formatar(media, erro):
    """'média ± erro' com 2 algarismos significativos no erro (GUM, JCGM 100:2008 §7.2.6)."""
    if not (np.isfinite(erro) and erro > 0):
        return f"{media:.4g}"
    casas = max(0, 1 - int(np.floor(np.log10(abs(erro)))))
    return f"{media:.{casas}f} ± {erro:.{casas}f}"


_SOBRESCRITO = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")


def unidade_mpl(unidade):
    """Unidade para o matplotlib: '⁻³' vira mathtext ($^{-3}$). O sinal sobrescrito '⁻' (U+207B)
    falta em várias fontes (a Aptos Serif, inclusive) e sai como um quadrado na figura."""
    return re.sub(r"⁻([⁰¹²³⁴⁵⁶⁷⁸⁹]+)",
                  lambda m: "$^{-" + m.group(1).translate(_SOBRESCRITO) + "}$", unidade or "")


def rotulo_media(ref):
    """Rótulo dos painéis: 'média ± EP (N réplicas): valor ± erro unidade'.

    EP = erro padrão da média: entre as médias das réplicas (a réplica é a unidade estatística);
    com uma réplica só, o SEM da série corrigido pela ineficiência estatística."""
    n = ref.get("n", np.nan)
    origem = (f"{int(n)} réplicas" if ref.get("replica") == "entre_replicas" and pd.notna(n)
              else "1 réplica")
    return (f"média ± EP ({origem}): {formatar(ref['media'], ref['sem'])} "
            f"{unidade_mpl(ref.get('unidade', ''))}").strip()

In [ ]:
tab = df_rdf.pivot(index="r_nm", columns="replica", values="g")
k = tab.shape[1]
if k > 1:
    media, dp, n_am, fonte_erro = tab.mean(axis=1).to_numpy(), tab.std(axis=1, ddof=1).to_numpy(), k, f"entre {k} réplicas"
else:
    bl = df_blocos.pivot(index="r_nm", columns="bloco", values="g")
    media, dp, n_am, fonte_erro = tab.iloc[:, 0].to_numpy(), bl.std(axis=1, ddof=1).to_numpy(), N_BLOCOS, f"entre {N_BLOCOS} blocos"
sem = dp / np.sqrt(n_am)
n_acum = df_rdf.pivot(index="r_nm", columns="replica", values="n_acumulado").mean(axis=1).to_numpy()
cont = df_rdf.pivot(index="r_nm", columns="replica", values="contagem").mean(axis=1).to_numpy()
df_perfil = pd.DataFrame({"r_nm": CENTROS, "g_medio": media, "desvio_padrao": dp, "sem": sem,
                          "ic_inferior": media - t_student(n_am - 1) * sem, "ic_superior": media + t_student(n_am - 1) * sem,
                          "n_acumulado": n_acum, "contagem_media": cont})

MIN_CONTAGENS_PICO, N_SIGMA_PICO = 30, 3.0
proeminencia = max(0.02, 0.10 * (np.nanmax(media) - np.nanmin(media)))
picos, props = find_peaks(media, prominence=proeminencia)
if picos.size:
    sigma = media[picos] / np.sqrt(np.maximum(cont[picos], 1.0))
    picos = picos[(cont[picos] >= MIN_CONTAGENS_PICO) & (props["prominences"] > N_SIGMA_PICO * sigma)]
patamar = float(np.median(media[CENTROS > 0.9 * BORDAS[-1]]))
i_pico = i_min = None
criterio = "—"
if picos.size:
    i_pico = int(picos[0])
    vales, _ = find_peaks(-media[i_pico:], prominence=max(0.01, 0.05 * (np.nanmax(media) - patamar)))
    if vales.size:
        i_min, criterio = int(i_pico + vales[0]), "mínimo local"
    else:
        cruza = np.flatnonzero(media[i_pico:] <= patamar)
        i_min, criterio = (int(i_pico + cruza[0]), "1º retorno ao patamar") if cruza.size else (None, "não encontrado")
else:
    print("[!] Nenhum pico sobreviveu aos filtros — veja a coluna contagem_media.")
g_cauda = float(media[CENTROS > 0.8 * BORDAS[-1]].mean())
n_coord = float(n_acum[i_min]) if i_min is not None else np.nan
volume_medio = float(np.mean(list(volumes.values())))
v_excluido = (volume_medio - df_selecao["grupo2"].mean() / DENSIDADE_REFERENCIA_NM3) if DENSIDADE_REFERENCIA_NM3 else np.nan

df_resumo = pd.DataFrame([{"fonte_do_erro": fonte_erro,
                           "r_primeiro_pico_nm": CENTROS[i_pico] if i_pico is not None else np.nan,
                           "g_primeiro_pico": media[i_pico] if i_pico is not None else np.nan,
                           "sem_primeiro_pico": sem[i_pico] if i_pico is not None else np.nan,
                           "r_primeiro_minimo_nm": CENTROS[i_min] if i_min is not None else np.nan,
                           "criterio_do_minimo": criterio, "numero_coordenacao": n_coord, "g_na_cauda": g_cauda,
                           "g_pico_renormalizado": media[i_pico] / g_cauda if i_pico is not None else np.nan,
                           "volume_caixa_nm3": volume_medio, "volume_excluido_nm3": v_excluido,
                           "massa_equivalente_kDa": v_excluido / 1.21}])
if abs(g_cauda - 1) > 0.05:
    print(f"[!] A cauda vale {g_cauda:.3f} (deveria tender a 1): densidade de referência errada, raio perto do "
          f"limite da caixa ou soluto grande no grupo 1 (aí a RDF radial não tem bulk a alcançar).")
if np.isfinite(v_excluido):
    print(f"Volume excluído pelo soluto: {v_excluido:.1f} nm³ (~{v_excluido / 1.21:.0f} kDa de proteína, a 1,21 nm³/kDa): "
          f"confira com a massa real; batendo, uma cauda abaixo de 1 é volume excluído, não erro.")

E = REFERENCIA_EXPERIMENTAL
df_experimento = pd.DataFrame()
if COMPARAR_EXPERIMENTO and i_pico is not None:
    df_experimento = pd.DataFrame([
        {"grandeza": "posição do 1º pico (nm)", "simulado": CENTROS[i_pico], "incerteza_sim": LARGURA_BIN_NM / 2,
         "experimental": E["r_pico_nm"], "incerteza_exp": E["u_r_pico_nm"]},
        {"grandeza": "altura do 1º pico", "simulado": media[i_pico], "incerteza_sim": sem[i_pico],
         "experimental": E["g_pico"], "incerteza_exp": E["u_g_pico"]},
        {"grandeza": "número de coordenação", "simulado": n_coord, "incerteza_sim": np.nan,
         "experimental": np.mean(COORDENACAO_EXP), "incerteza_exp": np.ptp(COORDENACAO_EXP) / 2}])
    df_experimento["desvio"] = df_experimento["simulado"] - df_experimento["experimental"]
    df_experimento["desvio_em_incertezas"] = df_experimento["desvio"] / np.hypot(
        df_experimento["incerteza_sim"].fillna(0), df_experimento["incerteza_exp"])
df_curva_exp = pd.read_csv(E["arquivo"]) if E.get("arquivo") else pd.DataFrame()

# ── Água pura do mesmo modelo (notebook 14) ─────────────────────────────────
M = REFERENCIA_MODELO
df_curva_modelo = pd.DataFrame()
if M and M.get("arquivo") and Path(M["arquivo"]).exists():
    d14 = pd.read_csv(M["arquivo"], comment="#", low_memory=False)
    df_curva_modelo = d14[d14["tabela"] == "rdf"].groupby("r_nm", as_index=False)["g"].mean()
    print(f"Água pura do modelo: {M['arquivo']} ({len(df_curva_modelo)} pontos de g(r))")
elif M and M.get("arquivo"):
    print(f"[modelo] {M['arquivo']} não existe: rode o notebook 14 ou ajuste o caminho; "
          "a figura usa só os valores digitados em REFERENCIA_MODELO.")
if M and len(df_experimento):
    df_experimento["agua_pura_modelo"] = [M["r_pico_nm"], M["g_pico"], M["numero_coordenacao"]]
    df_experimento["simulado_menos_modelo"] = df_experimento["simulado"] - df_experimento["agua_pura_modelo"]
if len(REPLICAS) == 1:
    print("[1 réplica] Não há linha 'entre_replicas': o erro é o SEM da própria série, "
          "corrigido pela ineficiência estatística (subestima a incerteza de observáveis lentos).")
display(df_resumo.T.rename(columns={0: "valor"}))
df_experimento

In [ ]:
import MDAnalysis as mda
u = mda.Universe("../rep1/md_300ns.tpr")
m_kda = u.select_atoms("protein").masses.sum() / 1000
print(f"proteína: {m_kda:.2f} kDa -> volume esperado {1.21 * m_kda:.1f} nm³")

## 5 · Gráfico (matplotlib)

**A** — $g(r)$ e $n(r)$ na faixa com estrutura (o formato da literatura). **B** — intervalo
inteiro, diagnóstico da cauda. **C** — comparação com o experimento (1º pico com barras de erro e,
se houver o CSV, a curva inteira).

In [ ]:
# ── Estilo geral (vale para todas as figuras deste notebook) ────────────────
FONTE = "Aptos Serif"                 # se não estiver instalada, usa DejaVu Serif
TAMANHO_FONTE = 8                     # pt
CORES = ["#003f5c", "#bc5090", "#ffa600", "#58508d", "#ff6361"]   # uma por réplica
COR_MEDIA = "#333333"                 # tracejado da média
COR_REFERENCIA = "#d62728"            # linhas de referência e faixa do descarte

if FONTE not in {f.name for f in font_manager.fontManager.ttflist}:
    FONTE = "DejaVu Serif"
plt.rcParams.update({
    "font.family": FONTE, "font.size": TAMANHO_FONTE,
    "axes.labelsize": TAMANHO_FONTE, "axes.titlesize": TAMANHO_FONTE,
    "xtick.labelsize": TAMANHO_FONTE - 1, "ytick.labelsize": TAMANHO_FONTE - 1,
    "legend.fontsize": TAMANHO_FONTE - 1, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.dpi": 120,
    "pdf.fonttype": 42,                # fontes embutidas como TrueType (editáveis)
    "svg.fonttype": "none",            # texto do SVG continua texto
})

In [ ]:
# ── Layout desta figura ─────────────────────────────────────────────────────
LARGURA, ALTURA = 7.0, 2.6
COR_G, COR_N, COR_EXP, COR_MODELO = "#003f5c", "#bc5090", "#ffa600", "#58508d"

# ── Gráfico ─────────────────────────────────────────────────────────────────
raio_fig = RAIO_FIGURA_NM or (min(4 * CENTROS[i_min], CENTROS[-1]) if i_min is not None else CENTROS[-1])
perto = CENTROS <= raio_fig
teto = 1.3 * 1.15 * float(media[cont >= MIN_CONTAGENS_PICO].max() if (cont >= MIN_CONTAGENS_PICO).any() else media.max())
fig, (ax_g, ax_c, ax_e) = plt.subplots(1, 3, figsize=(LARGURA, ALTURA), layout="constrained",
                                       gridspec_kw={"width_ratios": [1.3, 1, 1.1]})
ax_g.fill_between(CENTROS[perto], (media - sem)[perto], (media + sem)[perto], color=COR_G, alpha=0.3, lw=0)
ax_g.plot(CENTROS[perto], media[perto], color=COR_G, lw=1.0)
ax_g.axhline(1, color=COR_MEDIA, ls="--", lw=0.7)
ax_g.set(xlabel="r (nm)", ylabel="g(r)", xlim=(0, raio_fig), ylim=(0, teto))
ax_g.set_title(f"faixa: ± EP ({fonte_erro})", loc="right", fontsize=TAMANHO_FONTE - 1)
ax_n = ax_g.twinx()
ax_n.plot(CENTROS[perto], n_acum[perto], color=COR_N, lw=1.0)
ax_n.set_ylabel("n(r)", color=COR_N)
if i_min is not None:
    ax_g.axvline(CENTROS[i_min], color=COR_REFERENCIA, ls=":", lw=0.8)
    ax_n.annotate(f"n = {n_coord:.2f}", (0.97, 0.06), xycoords="axes fraction", ha="right", color=COR_N,
                  fontsize=TAMANHO_FONTE - 1)

ax_c.plot(CENTROS, media, color=COR_G, lw=0.8)
ax_c.axhline(1, color=COR_MEDIA, ls="--", lw=0.7)
ax_c.set(xlabel="r (nm)", ylabel="g(r)", xlim=(0, CENTROS[-1]), ylim=(0, teto))
ax_c.set_title(f"cauda: {g_cauda:.3f}", loc="right", fontsize=TAMANHO_FONTE - 1)

ax_e.fill_between(CENTROS, df_perfil["ic_inferior"], df_perfil["ic_superior"], color=COR_G, alpha=0.2, lw=0)
ax_e.plot(CENTROS, media, color=COR_G, lw=0.9,
          label=f"sistema, bulk (> {DISTANCIA_EXCLUSAO_NM:g} nm)" if SELECAO_EXCLUSAO else "sistema")
if len(df_curva_modelo):
    ax_e.plot(df_curva_modelo["r_nm"], df_curva_modelo["g"], color=COR_MODELO, lw=0.9, ls="--", label=M["rotulo"])
elif M:
    ax_e.errorbar(M["r_pico_nm"], M["g_pico"], yerr=M["u_g_pico"], fmt="s", ms=3, color=COR_MODELO, capsize=2,
                  lw=0.8, label=f"{M['rotulo']}: 1º pico")
if len(df_curva_exp):
    ax_e.plot(df_curva_exp["r_nm"], df_curva_exp["g"], color=COR_EXP, lw=0.9, label=E["rotulo"])
ax_e.errorbar(E["r_pico_nm"], E["g_pico"], xerr=E["u_r_pico_nm"], yerr=E["u_g_pico"], fmt="o", ms=3.5,
              color=COR_N, capsize=2, lw=0.8, label=f"{E['rotulo']}: 1º pico")
ax_e.set(xlabel="r (nm)", ylabel="g$_{OO}$(r)", xlim=(0.2, 0.8), ylim=(0, teto))
ax_e.legend(frameon=True, fontsize=TAMANHO_FONTE - 2)
for eixo, letra in zip((ax_g, ax_c, ax_e), "ABC"):
    eixo.set_title(letra, loc="left", fontweight="bold")
fig_rdf = fig
plt.show()

## 6 · Gráfico interativo (Plotly)

In [ ]:
# ── Layout ──────────────────────────────────────────────────────────────────
ALTURA_PX = 450                # pixels
LARGURA_PX = None              # None = largura da página
TEMPLATE = "simple_white"
TAMANHO_FONTE_PX = 13

# ── Gráfico ─────────────────────────────────────────────────────────────────
# legenda: só as réplicas; média, n(r) (eixo direito) e o ponto experimental estão rotulados no gráfico
fig_i = make_subplots(specs=[[{"secondary_y": True}]])
fig_i.add_trace(go.Scatter(x=np.r_[CENTROS, CENTROS[::-1]], y=np.r_[df_perfil["ic_superior"], df_perfil["ic_inferior"][::-1]],
                           fill="toself", fillcolor="rgba(51,51,51,0.15)", line=dict(width=0), hoverinfo="skip",
                           showlegend=False), secondary_y=False)
for i, (rep, g) in enumerate(df_rdf.groupby("replica", sort=False)):
    fig_i.add_trace(go.Scatter(x=g["r_nm"], y=g["g"], mode="lines", name=rep,
                               line=dict(color=CORES[i % len(CORES)], width=1)), secondary_y=False)
fig_i.add_trace(go.Scatter(x=CENTROS, y=media, mode="lines", name="g(r) média", showlegend=False,
                           line=dict(color=COR_MEDIA, width=2), customdata=np.c_[sem, cont],
                           hovertemplate="média — r = %{x:.3f} nm<br>g = %{y:.3f} ± %{customdata[0]:.3f}"
                                         "<br>pares no bin: %{customdata[1]:.0f}<extra></extra>"), secondary_y=False)
fig_i.add_trace(go.Scatter(x=CENTROS, y=n_acum, mode="lines", name="n(r)", showlegend=False,
                           line=dict(color=COR_MEDIA, width=1.2, dash="dashdot")), secondary_y=True)
if len(df_curva_modelo):
    fig_i.add_trace(go.Scatter(x=df_curva_modelo["r_nm"], y=df_curva_modelo["g"], mode="lines", name=M["rotulo"],
                               showlegend=False, line=dict(color="#58508d", dash="dash", width=1.5)), secondary_y=False)
fig_i.add_trace(go.Scatter(x=[E["r_pico_nm"]], y=[E["g_pico"]], mode="markers", name=E["rotulo"], showlegend=False,
                           marker=dict(color=COR_REFERENCIA, size=8),
                           error_x=dict(array=[E["u_r_pico_nm"]]), error_y=dict(array=[E["u_g_pico"]])), secondary_y=False)
fig_i.add_annotation(x=E["r_pico_nm"], y=E["g_pico"], text=f"{E['rotulo']}: 1º pico", showarrow=False,
                     xanchor="left", yanchor="bottom", xshift=6, font=dict(color=COR_REFERENCIA))
fig_i.add_annotation(x=0, y=1, xref="x domain", yref="y domain", xanchor="left", yanchor="top", showarrow=False,
                     text=f"linha escura: g(r) média ± IC {NIVEL_CONFIANCA:.0%}; traço-ponto: n(r)"
                          + (f"; tracejado roxo: {M['rotulo']}" if len(df_curva_modelo) else ""))
fig_i.add_hline(y=1, line=dict(color=COR_MEDIA, dash="dash", width=1))
fig_i.update_xaxes(title_text="r (nm)", range=[0, raio_fig])
fig_i.update_yaxes(title_text="g(r)", range=[0, teto], secondary_y=False)
fig_i.update_yaxes(title_text="n(r) (traço-ponto)", secondary_y=True, showgrid=False)
fig_i.update_layout(template=TEMPLATE, height=ALTURA_PX, width=LARGURA_PX, legend_title_text="Réplica",
                    font=dict(family=FONTE, size=TAMANHO_FONTE_PX))
fig_interativo = fig_i
fig_i.show()

## 7 · Exportação

Tudo vai para `DIR_SAIDA`, separado por formato:

```
{DIR_SAIDA}/
├── png/   600 dpi
├── pdf/   vetorial, fontes embutidas
├── svg/   vetorial, texto editável
├── html/  gráficos interativos (Plotly)
└── <prefixo>_dados.csv   ← CSV ÚNICO com todos os dados
```

O CSV é **um só**, em formato longo: a coluna `tabela` diz de que tabela vem
cada linha; as colunas que não se aplicam ficam vazias. As linhas iniciadas por
`#` registram data, hora, versões e parâmetros. Para ler:
`pd.read_csv(arquivo, comment="#")`.

| `tabela` | conteúdo |
|---|---|
| `perfil` | g(r) médio ± SEM e IC, n(r), contagem média de pares por bin |
| `por_replica` | g(r), contagem e n(r) de cada réplica |
| `por_bloco` | g(r) por bloco da produção |
| `resumo` | 1º pico, 1º mínimo, número de coordenação, cauda, volume excluído |
| `comparacao_experimental` | simulado × experimental (posição, altura, coordenação) com desvio em incertezas |
| `selecao` | átomos, pares por frame, passo de leitura e altura mínima da caixa por réplica |

In [ ]:
FORMATOS = ("png", "pdf", "svg")
DPI = 600
FUNDO_TRANSPARENTE = False
PLOTLY_JS = "cdn"                  # "cdn" = HTML leve; True = embute o plotly.js (abre offline)
EXPORTAR_PLOTLY_ESTATICO = False   # True = também PNG/PDF/SVG do Plotly (precisa do kaleido)

FIGURAS = {"rdf": fig_rdf}
INTERATIVOS = {"rdf": fig_interativo}
TABELAS = {"perfil": df_perfil, "por_replica": df_rdf, "por_bloco": df_blocos, "resumo": df_resumo,
          "comparacao_experimental": df_experimento, "selecao": df_selecao, "agua_pura_modelo": df_curva_modelo}
PARAMETROS = {"descarte_ns": DESCARTE_NS, "dt_analise_ps": DT_ANALISE_PS, "grupo1": SELECAO_GRUPO_1,
              "grupo2": SELECAO_GRUPO_2, "max_grupo1": MAX_ATOMOS_GRUPO_1, "largura_bin_nm": LARGURA_BIN_NM,
              "normalizacao": NORMALIZACAO, "densidade_referencia_nm3": DENSIDADE_REFERENCIA_NM3,
              "exclusao": SELECAO_EXCLUSAO, "distancia_exclusao_nm": DISTANCIA_EXCLUSAO_NM}

# ── Figuras ─────────────────────────────────────────────────────────────────
for nome, fig in FIGURAS.items():
    for ext in FORMATOS:
        (DIR_SAIDA / ext).mkdir(parents=True, exist_ok=True)
        fig.savefig(DIR_SAIDA / ext / f"{nome}.{ext}", dpi=DPI, bbox_inches="tight",
                    transparent=FUNDO_TRANSPARENTE)
for nome, fig in INTERATIVOS.items():
    (DIR_SAIDA / "html").mkdir(parents=True, exist_ok=True)
    fig.write_html(DIR_SAIDA / "html" / f"{nome}.html", include_plotlyjs=PLOTLY_JS,
                   config={"displaylogo": False, "toImageButtonOptions": {"format": "svg"}})
    if EXPORTAR_PLOTLY_ESTATICO:
        for ext in FORMATOS:
            try:
                fig.write_image(DIR_SAIDA / ext / f"{nome}_interativo.{ext}", scale=3)
            except Exception as erro:
                print(f"[aviso] {nome}.{ext} do Plotly não exportado: {erro}")
                break

# ── CSV único ───────────────────────────────────────────────────────────────
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
partes = []
for nome, tabela in TABELAS.items():
    t = tabela.copy()
    t.insert(0, "tabela", nome)
    partes.append(t)
dados = pd.concat(partes, ignore_index=True, sort=False)
arquivo_csv = DIR_SAIDA / f"{PREFIXO}_dados.csv"
with open(arquivo_csv, "w", encoding="utf-8", newline="") as f:
    f.write(f"# {TITULO}\n")
    f.write(f"# gerado em {INICIO:%Y-%m-%d %H:%M:%S} | sistema {SISTEMA} | réplicas {REPLICAS}\n")
    f.write("# versoes: " + "; ".join(f"{k} {v}" for k, v in VERSOES.items()) + "\n")
    f.write("# parametros: " + "; ".join(f"{k} = {v}" for k, v in PARAMETROS.items()) + "\n")
    f.write("# tabelas: " + ", ".join(f"{k} ({len(v)} linhas)" for k, v in TABELAS.items()) + "\n")
    dados.to_csv(f, index=False)

# ── O que foi gravado ───────────────────────────────────────────────────────
df_exportados = pd.DataFrame([{"pasta": str(p.parent.relative_to(DIR_SAIDA)), "arquivo": p.name,
                               "tamanho_kB": round(p.stat().st_size / 1024, 1)}
                              for p in sorted(DIR_SAIDA.rglob("*")) if p.is_file()])
print(f"Resultados em: {DIR_SAIDA.resolve()}")
print(f"Duração da execução: {datetime.now() - INICIO}")
df_exportados